In [1]:
%%configure -f
{
  "driverMemory": "512M",
  "executorMemory": "512M",
  "conf": {
    "spark.driver.memoryOverhead": "128M",
    "spark.executor.memoryOverhead": "128M"
  }
}

In [2]:
from pyspark.sql import SparkSession 
from pyspark.sql.types import StructType, StructField, StringType, FloatType, TimestampType 
from pyspark.sql.functions import col, to_timestamp, current_timestamp

# Określenie struktury pliku wejściowego oraz typów danych
schema = StructType([ StructField("timestamp", StringType(), True),
    StructField("station_id", StringType(), True), 
    StructField("temperature", FloatType(), True), 
    StructField("humidity", FloatType(), True),
    StructField("pressure", FloatType(), True), 
    StructField("wind_speed", FloatType(), True), 
    StructField("wind_direction", FloatType(), True), 
    StructField("rain_mm", FloatType(), True), 
    StructField("cloud_cover", FloatType(), True) 
])

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,application_1782669383371_0002,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
# Czytanie danych ze schematem
input_stream = spark.readStream.schema(schema).json("s3://emr-notebook-as-2137/raw/")
# Przetworzenie czasu pobrania na timestamp oraz dodanie czasu przetworzenia
processed_stream = input_stream.withColumn("event_time", to_timestamp(col("timestamp"))).withColumn("processing_time", current_timestamp()).drop("timestamp")
# Dodawanie kolejnych danych do bucketa w formacie parquet
query = processed_stream.writeStream.format("parquet").option("path", "s3://emr-notebook-as-2137/curated_nrt/").option("checkpointLocation", "s3://emr-notebook-as-2137/checkpoints/nrt_processing/").outputMode("append").start()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…